# LPatchTST — Kaggle 2x T4 GPU Notebook

This notebook handles repository setup, data preparation, multi-GPU DDP training, and downstream evaluation on Kaggle.

### Pre-requisites:
1. **Internet** must be turned **ON** in the right-hand settings panel.
2. **Accelerator** must be set to **GPU T4 x2**.

In [ ]:
%%bash
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  CELL 1 — Clone Repo & Set Up CSV Dataset                           ║
# ╚══════════════════════════════════════════════════════════════════════╝
REPO_DIR="/kaggle/working/Lpatchtst"

if [ -d "$REPO_DIR" ]; then
    echo "Repository already exists. Updating to latest main..."
    cd "$REPO_DIR"
    git fetch origin
    git reset --hard origin/main
    git submodule update --init --recursive
else
    echo "Cloning repository (including submodules)..."
    git clone --recurse-submodules https://github.com/ayan1-git/Lpatchtst "$REPO_DIR"
fi

# Install lightweight required libraries (torch/numpy/pandas are pre-installed)
pip install -q einops safetensors scikit-learn

cd "$REPO_DIR"
mkdir -p Data

# Check if the cloned repository already contains CSV files in Data/
csv_count=$(ls Data/*.csv 2>/dev/null | wc -l)

if [ "$csv_count" -gt 0 ]; then
    echo "Found $csv_count pre-packaged CSV files in Data/ directory (cloned from Git)."
    echo "Using pre-packaged dataset for training."
    ls -lh Data/
else
    # Auto-discover any Kaggle input folder containing CSV files and link them
    echo "No CSV files found in cloned Data/ folder. Searching under /kaggle/input/..."
    FOUND=0
    for d in /kaggle/input/*; do
        if [ -d "$d" ]; then
            if ls "$d"/*.csv >/dev/null 2>&1; then
                echo "-> Found CSV files in $d. Symlinking to Data/..."
                ln -sf "$d"/*.csv Data/
                FOUND=1
            fi
        fi
    done
    if [ $FOUND -eq 0 ]; then
        echo "⚠️ Warning: No CSV files found. Please copy your CSVs to $REPO_DIR/Data/."
    else
        echo "Data folder contents successfully mapped from Kaggle inputs:"
        ls -lh Data/
    fi
fi
echo "✅ Setup complete!"

In [ ]:
%%bash
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  CELL 2 — Run DDP Training on 2x T4 GPUs via torchrun               ║
# ╚══════════════════════════════════════════════════════════════════════╝
cd /kaggle/working/Lpatchtst

# Automatically detect how many GPUs are active (e.g. 2 for T4x2)
NUM_GPUS=$(python3 -c "import torch; print(torch.cuda.device_count())")
echo "Launching distributed training on $NUM_GPUS GPUs..."
echo "----------------------------------------------------------------------"

torchrun --standalone --nnodes=1 --nproc_per_node=$NUM_GPUS --master_port=29500 train_pretrain_fine.py

In [ ]:
%%bash
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  CELL 3 — Downstream Evaluation & Diagnostics                       ║
# ╚══════════════════════════════════════════════════════════════════════╝
cd /kaggle/working/Lpatchtst

echo "Launching single-process walk-forward evaluation..."
echo "----------------------------------------------------------------------"

python3 audit_scripts/batch_eval.py